In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.preprocessing import StandardScaler


df = pd.read_csv("../data/raw/raw_data.csv")
df.head()

/tmp/ipykernel_6303/1828962278.py:9: DtypeWarning: Columns (0: id, 1: desc, 2: next_pymnt_d, 3: verification_status_joint, 4: sec_app_earliest_cr_line, 5: hardship_type, 6: hardship_reason, 7: hardship_status, 8: hardship_start_date, 9: hardship_end_date, 10: payment_plan_start_date, 11: hardship_loan_status, 12: debt_settlement_flag_date, 13: settlement_status, 14: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/raw_data.csv")


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Impute 'Informative Nulls' with a Constant (Fixed Value) #

# NaN in 'months since...' negative events usually means the event NEVER happened. 
# We impute with 999 so the model treats it as a distinct, safe category.
month_null_cols = [
    'mths_since_last_delinq', 
    'mths_since_last_record',
    'mths_since_recent_bc_dlq', 
    'mths_since_last_major_derog',
    'mths_since_recent_revol_delinq',
    'mths_since_recent_inq' # Inquiries are events, so 999 (no recent inquiries) is fine
]

for col in month_null_cols:
    if col in df.columns:
        df[col] = df[col].fillna(999)
        
# Nan in 'Number of...' usually means the event NEVER happened.
# We impute with 0 because it is a count and 0 makes sense as a value.
number_null_cols = [
    'delinq_2yrs',
    'inq_last_6mths',
    'open_acc',
    'pub_rec',
    'total_acc',
    'acc_now_delinq',
    'tot_coll_amt',
    'open_acc_6m',
    'open_act_il',
    'open_il_24m',
    'inq_fi',
    'total_cu_tl',
    'inq_last_12m',
    'acc_open_past_24mths',
    'chargeoff_within_12_mths',
    'mort_acc',
    'num_accts_ever_120_pd',
    'num_actv_bc_tl',
    'num_actv_rev_tl',
    'num_bc_sats',
    'num_bc_tl',
    'num_il_tl',
    'num_op_rev_tl',
    'num_rev_accts',
    'num_rev_tl_bal_gt_0',
    'num_sats',
    'pub_rec_bankruptcies',
    'tax_liens'
]

for col in number_null_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Impute 'Informative Nulls' with the Mode (Most Frequent Value) - Only one categorical column left
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].mode()[0])

# Impute 'Missing at Random' with the Median (Central Tendency) - Only numerical columns left
cols_with_nulls = df.columns[df.isnull().any()].tolist()

for col in cols_with_nulls:
    median_val = df[col].median(numeric_only=True)
    df[col] = df[col].fillna(median_val)